# Player Points Context Analysis

This notebook explores whether player scoring is related to a richer set of pre-game context features than opponent net rating alone.

Main ideas tested:

- opponent rolling defensive rating over previous `20` team games
- opponent rolling pace over previous `20` team games
- player rolling average points over previous `40` player games
- player rolling average minutes over previous `20` player games
- player rolling average usage percentage over previous `20` player games
- home vs away

The core target is:

- `points_minus_player_roll40 = actual points - player rolling average points`

That lets us ask whether opponent/team-context variables help explain whether a player scored above or below his own baseline.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd()
PLAYERS_PATH = PROJECT_ROOT / "data/box_scores/players.csv"
TEAMS_PATH = PROJECT_ROOT / "data/box_scores/teams.csv"

TEAM_WINDOW = 20
PLAYER_POINTS_WINDOW = 40
PLAYER_ROLE_WINDOW = 20

MIN_TEAM_HISTORY = TEAM_WINDOW
MIN_PLAYER_POINTS_HISTORY = PLAYER_POINTS_WINDOW
MIN_PLAYER_ROLE_HISTORY = PLAYER_ROLE_WINDOW


In [ ]:
def parse_minutes(value):
    if pd.isna(value) or value == "":
        return np.nan
    text = str(value)
    if ":" in text:
        minutes, seconds = text.split(":", 1)
        return float(minutes) + float(seconds) / 60.0
    try:
        return float(text)
    except ValueError:
        return np.nan


def load_data(players_path=PLAYERS_PATH, teams_path=TEAMS_PATH):
    players = pd.read_csv(players_path, dtype={"gameId": str, "teamId": str, "personId": str})
    teams = pd.read_csv(teams_path, dtype={"gameId": str, "teamId": str})

    for df in (players, teams):
        df["gameId"] = df["gameId"].str.lstrip("0")
        df["gameDate"] = pd.to_datetime(df["gameDate"])

    players["minutes_float"] = players["minutes"].map(parse_minutes)
    players["isHomeInt"] = players["isHome"].astype(int)

    return players, teams


def build_analysis_frame():
    players, teams = load_data()

    teams = teams.sort_values(["teamId", "gameDate", "gameId"]).copy()
    teams["opponent_defensive_rating_roll20"] = (
        teams.groupby("teamId")["defensiveRating"]
        .transform(lambda s: s.shift(1).rolling(TEAM_WINDOW, min_periods=MIN_TEAM_HISTORY).mean())
    )
    teams["opponent_pace_roll20"] = (
        teams.groupby("teamId")["pace"]
        .transform(lambda s: s.shift(1).rolling(TEAM_WINDOW, min_periods=MIN_TEAM_HISTORY).mean())
    )

    opponent_lookup = teams[[
        "gameId",
        "teamId",
        "teamTricode",
        "opponent_defensive_rating_roll20",
        "opponent_pace_roll20",
    ]].rename(columns={
        "teamId": "opponentTeamId",
        "teamTricode": "opponentTeamTricode",
    })

    analysis = players.merge(opponent_lookup, on="gameId", how="left")
    analysis = analysis[analysis["teamId"] != analysis["opponentTeamId"]].copy()
    analysis = analysis.sort_values(["personId", "gameDate", "gameId"]).copy()

    analysis["player_points_roll40"] = (
        analysis.groupby("personId")["points"]
        .transform(lambda s: s.shift(1).rolling(PLAYER_POINTS_WINDOW, min_periods=MIN_PLAYER_POINTS_HISTORY).mean())
    )
    analysis["player_minutes_roll20"] = (
        analysis.groupby("personId")["minutes_float"]
        .transform(lambda s: s.shift(1).rolling(PLAYER_ROLE_WINDOW, min_periods=MIN_PLAYER_ROLE_HISTORY).mean())
    )
    analysis["player_usage_roll20"] = (
        analysis.groupby("personId")["usagePercentage"]
        .transform(lambda s: s.shift(1).rolling(PLAYER_ROLE_WINDOW, min_periods=MIN_PLAYER_ROLE_HISTORY).mean())
    )

    analysis["player_name"] = analysis["firstName"] + " " + analysis["familyName"]
    analysis["points_minus_player_roll40"] = analysis["points"] - analysis["player_points_roll40"]
    analysis["season"] = analysis["gameDate"].apply(lambda d: f"{d.year}-{str(d.year + 1)[2:]}" if d.month >= 7 else f"{d.year - 1}-{str(d.year)[2:]}")

    return analysis


In [ ]:
analysis = build_analysis_frame()
analysis_ready = analysis.dropna(subset=[
    "opponent_defensive_rating_roll20",
    "opponent_pace_roll20",
    "player_points_roll40",
    "player_minutes_roll20",
    "player_usage_roll20",
]).copy()

print("All player-game rows:", len(analysis))
print("Rows with full rolling history:", len(analysis_ready))
print("Unique players with full rolling history:", analysis_ready["personId"].nunique())

analysis_ready[[
    "gameDate",
    "season",
    "player_name",
    "teamTricode",
    "opponentTeamTricode",
    "points",
    "player_points_roll40",
    "points_minus_player_roll40",
    "player_minutes_roll20",
    "player_usage_roll20",
    "opponent_defensive_rating_roll20",
    "opponent_pace_roll20",
    "isHomeInt",
]].head()


In [ ]:
feature_cols = [
    "opponent_defensive_rating_roll20",
    "opponent_pace_roll20",
    "player_minutes_roll20",
    "player_usage_roll20",
    "isHomeInt",
]

corr_summary = pd.DataFrame({
    "feature": feature_cols,
    "corr_with_points": [analysis_ready[col].corr(analysis_ready["points"]) for col in feature_cols],
    "corr_with_points_minus_roll40": [analysis_ready[col].corr(analysis_ready["points_minus_player_roll40"]) for col in feature_cols],
})

corr_summary.sort_values("corr_with_points_minus_roll40", ascending=False)


In [ ]:
X = analysis_ready[feature_cols].astype(float).to_numpy()
y = analysis_ready["points_minus_player_roll40"].astype(float).to_numpy()

X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
y_mean = y.mean()
y_std = y.std()

Xz = (X - X_mean) / X_std
yz = (y - y_mean) / y_std

X_design = np.column_stack([np.ones(len(Xz)), Xz])
coef, *_ = np.linalg.lstsq(X_design, yz, rcond=None)
pred = X_design @ coef
r2 = 1.0 - ((yz - pred) ** 2).sum() / ((yz - yz.mean()) ** 2).sum()

beta_summary = pd.DataFrame({
    "feature": feature_cols,
    "standardized_beta": coef[1:],
}).sort_values("standardized_beta", ascending=False)

print(f"Linear fit R^2 for points_minus_player_roll40: {r2:.4f}")
beta_summary


In [ ]:
high_minute = analysis_ready[analysis_ready["player_minutes_roll20"] >= 24].copy()
print("High-minute subset rows:", len(high_minute))
print("High-minute subset players:", high_minute["personId"].nunique())

high_minute_corrs = pd.DataFrame({
    "feature": feature_cols,
    "corr_with_points_minus_roll40": [high_minute[col].corr(high_minute["points_minus_player_roll40"]) for col in feature_cols],
})

high_minute_corrs.sort_values("corr_with_points_minus_roll40", ascending=False)


In [ ]:
def player_corr_summary(df, min_games=80, min_avg_points=12.0, min_minutes=24.0):
    rows = []
    for (person_id, player_name), player_df in df.groupby(["personId", "player_name"]):
        player_df = player_df.sort_values(["gameDate", "gameId"])
        if len(player_df) < min_games:
            continue
        if player_df["points"].mean() < min_avg_points:
            continue
        if player_df["player_minutes_roll20"].mean() < min_minutes:
            continue

        rows.append({
            "personId": person_id,
            "player_name": player_name,
            "games": len(player_df),
            "avg_points": player_df["points"].mean(),
            "corr_opp_def": player_df["points_minus_player_roll40"].corr(player_df["opponent_defensive_rating_roll20"]),
            "corr_opp_pace": player_df["points_minus_player_roll40"].corr(player_df["opponent_pace_roll20"]),
        })

    return pd.DataFrame(rows)


player_summary = player_corr_summary(analysis_ready)
print("Players in player-level summary:", len(player_summary))
player_summary.head()


In [ ]:
print("Top positive player-level correlations with opponent defensive rating")
display(player_summary.sort_values("corr_opp_def", ascending=False).head(15))

print("Top positive player-level correlations with opponent pace")
display(player_summary.sort_values("corr_opp_pace", ascending=False).head(15))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(player_summary["corr_opp_def"].dropna(), bins=30, edgecolor="white")
axes[0].axvline(player_summary["corr_opp_def"].median(), color="red", linestyle="--", label="median")
axes[0].set_title("Player-Level Correlations: Opponent Defensive Rating")
axes[0].set_xlabel("corr(points - player roll40, opponent def roll20)")
axes[0].set_ylabel("Player count")
axes[0].legend()

axes[1].hist(player_summary["corr_opp_pace"].dropna(), bins=30, edgecolor="white")
axes[1].axvline(player_summary["corr_opp_pace"].median(), color="red", linestyle="--", label="median")
axes[1].set_title("Player-Level Correlations: Opponent Pace")
axes[1].set_xlabel("corr(points - player roll40, opponent pace roll20)")
axes[1].set_ylabel("Player count")
axes[1].legend()

plt.tight_layout()


## Notes

- Opponent rolling defensive rating and opponent rolling pace usually look more directly relevant to player points than opponent net rating.
- The likely interpretation is not that these variables are huge on their own, but that they provide modest context around a player's own baseline.
- If the coefficients/correlations look stable, the next step would be to join this feature table to the betting lines and benchmark whether these context variables help versus the simple baseline.
